# Club Chatbot: Memory That Survives a Restart

This Colab notebook builds a topic-locked club chatbot with LangGraph, a Supabase Postgres checkpointer, and a Gradio interface. Run the cells from top to bottom.

> **Secrets:** Add `OPENAI_API_KEY` and `SUPABASE_DB_URL` to Colab Secrets and enable notebook access for both. Use the Supabase **Session Pooler** connection URL. Never print either secret or put it in code or GitHub. The temporary public Gradio link lets anyone who has it use your API key; do not post the link publicly and stop the app when you finish.

In [ ]:
%pip -q install langgraph langchain langchain-openai langgraph-checkpoint-postgres 'psycopg[binary]' gradio

In [ ]:
import os
import uuid

import gradio as gr
import psycopg
from google.colab import userdata
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from psycopg.rows import dict_row
from typing import Annotated, TypedDict

openai_key = userdata.get("OPENAI_API_KEY")
database_url = userdata.get("SUPABASE_DB_URL")
if not openai_key or not database_url:
    raise ValueError("Add OPENAI_API_KEY and SUPABASE_DB_URL to Colab Secrets and enable notebook access.")
os.environ["OPENAI_API_KEY"] = openai_key

## Connect Supabase and create the checkpoint tables

`PostgresSaver.setup()` creates the LangGraph checkpoint tables. The small `club_conversations` table stores a readable topic and preview for the Gradio saved-conversation list.

In [ ]:
conn = psycopg.connect(
    database_url,
    autocommit=True,
    row_factory=dict_row,
    prepare_threshold=0,
)
checkpointer = PostgresSaver(conn)
checkpointer.setup()

conn.execute("""
    CREATE TABLE IF NOT EXISTS club_conversations (
        thread_id TEXT PRIMARY KEY,
        topic TEXT NOT NULL,
        preview TEXT NOT NULL DEFAULT 'New conversation',
        updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")
print("Supabase checkpoint storage is ready.")

## Define and draw the graph

The system message is built from the current `topic` on every call. It is sent to the model but is not stored in `messages`. The topic has no reducer, so the newest selected topic replaces the previous one.

In [ ]:
TOPICS = [
    "Astronomy",
    "Cooking",
    "Football",
    "World Geography",
    "Python Programming",
]
model = init_chat_model("openai:gpt-6-luna")

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str

def chatbot(state: State):
    topic = state["topic"]
    SYSTEM = SystemMessage(
        content=(
            f"You are a friendly tutor in the {topic} club. Answer only questions about {topic}. "
            f"If asked about another topic, politely say you can help only with {topic} and suggest "
            f"switching clubs. This includes questions about any club selected earlier in this conversation."
        )
    )
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=checkpointer)
display(Image(graph.get_graph().draw_mermaid_png()))

## Store and load conversations

The graph checkpoint stores messages and topic under a conversation's `thread_id`. The metadata table makes saved conversations findable after the Colab runtime restarts.

In [ ]:
def save_new_conversation(thread_id: str, topic: str):
    conn.execute(
        "INSERT INTO club_conversations (thread_id, topic) VALUES (%s, %s)",
        (thread_id, topic),
    )

def update_conversation(thread_id: str, topic: str, preview: str | None = None):
    if preview is None:
        conn.execute(
            "UPDATE club_conversations SET topic = %s, updated_at = NOW() WHERE thread_id = %s",
            (topic, thread_id),
        )
    else:
        conn.execute(
            "UPDATE club_conversations SET topic = %s, preview = %s, updated_at = NOW() WHERE thread_id = %s",
            (topic, preview, thread_id),
        )

def list_conversations():
    rows = conn.execute(
        "SELECT thread_id, topic, preview FROM club_conversations ORDER BY updated_at DESC"
    ).fetchall()
    return [(f"{row['topic']} — {row['preview']}", row["thread_id"]) for row in rows]

def preview_from(messages):
    first_question = next(
        (message.content.strip() for message in messages if message.type == "human"),
        "New conversation",
    )
    return " ".join(first_question.split())[:60]

def to_chat_messages(messages):
    return [
        {"role": "user" if message.type == "human" else "assistant", "content": message.content}
        for message in messages
        if message.type in ("human", "ai")
    ]

## Build the Gradio interface

Changing the club keeps the same thread and chat. Starting a new conversation creates a new UUID. Saved conversations are queried from Supabase, not held in a Python history variable.

In [ ]:
def start_app():
    thread_id = str(uuid.uuid4())
    topic = TOPICS[0]
    save_new_conversation(thread_id, topic)
    return topic, [], thread_id, gr.update(choices=list_conversations(), value=thread_id)

def new_conversation(topic):
    thread_id = str(uuid.uuid4())
    save_new_conversation(thread_id, topic)
    return [], thread_id, gr.update(choices=list_conversations(), value=thread_id)

def change_topic(topic, thread_id):
    if thread_id:
        config = {"configurable": {"thread_id": thread_id}}
        snapshot = graph.get_state(config)
        if snapshot.values:
            graph.update_state(config, {"topic": topic})
        update_conversation(thread_id, topic)
        return gr.update(choices=list_conversations(), value=thread_id)
    return gr.update(choices=list_conversations())

def open_conversation(thread_id):
    if not thread_id:
        return gr.update(), [], None
    config = {"configurable": {"thread_id": thread_id}}
    saved = graph.get_state(config).values
    row = conn.execute(
        "SELECT topic FROM club_conversations WHERE thread_id = %s", (thread_id,)
    ).fetchone()
    topic = row["topic"] if row else saved.get("topic", TOPICS[0])
    messages = saved.get("messages", [])
    return topic, to_chat_messages(messages), thread_id

def send_message(text, topic, thread_id):
    if not text or not text.strip():
        return "", gr.update(), gr.update(), thread_id
    if not thread_id:
        thread_id = str(uuid.uuid4())
        save_new_conversation(thread_id, topic)
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "topic": topic},
        config,
    )
    messages = result["messages"]
    update_conversation(thread_id, topic, preview_from(messages))
    return "", to_chat_messages(messages), gr.update(choices=list_conversations(), value=thread_id), thread_id

with gr.Blocks(title="Curious Minds Club Chatbot") as demo:
    gr.Markdown("# Curious Minds Club Chatbot\nChoose a club and ask a question. Your saved conversations are stored in Supabase.")
    thread_state = gr.State()
    topic_dropdown = gr.Dropdown(TOPICS, value=TOPICS[0], label="Club topic")
    saved_dropdown = gr.Dropdown(choices=[], label="Saved conversations")
    new_button = gr.Button("New conversation")
    chat = gr.Chatbot(label="Chat")
    message_box = gr.Textbox(label="Your message", placeholder="Ask about the selected club...")
    send_button = gr.Button("Send")

    demo.load(start_app, outputs=[topic_dropdown, chat, thread_state, saved_dropdown])
    topic_dropdown.change(change_topic, inputs=[topic_dropdown, thread_state], outputs=saved_dropdown)
    saved_dropdown.change(open_conversation, inputs=saved_dropdown, outputs=[topic_dropdown, chat, thread_state])
    new_button.click(new_conversation, inputs=topic_dropdown, outputs=[chat, thread_state, saved_dropdown])
    send_button.click(send_message, inputs=[message_box, topic_dropdown, thread_state], outputs=[message_box, chat, saved_dropdown, thread_state])
    message_box.submit(send_message, inputs=[message_box, topic_dropdown, thread_state], outputs=[message_box, chat, saved_dropdown, thread_state])

demo.launch(share=True)

## Manual test: topic switching and persistence

Run this cell to check the graph without the interface. It uses a separate test thread. Then test the same behavior in Gradio and restart the Colab runtime to confirm that saved conversations still load.

In [ ]:
test_config = {"configurable": {"thread_id": "manual-test-1"}}

def ask(text, topic):
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "topic": topic},
        test_config,
    )
    print(f"[{topic}] {text}\n{result['messages'][-1].content}\n")

ask("How do I make pancakes?", "Cooking")
ask("Who won the 2022 World Cup?", "Cooking")  # should politely refuse
ask("Who won the 2022 World Cup?", "Football")  # same thread, new topic
ask("How long should I fry each pancake?", "Football")  # should refuse the old topic
print("Saved graph state:", graph.get_state(test_config).values)

## Before submitting

1. Try each club, ask an unrelated question, switch clubs mid-conversation, and ask about the old club again.
2. Start a conversation, restart the Colab runtime, reopen it from **Saved conversations**, and confirm the topic and messages return.
3. Inspect the Supabase Table Editor for the LangGraph checkpoint tables and `club_conversations`.
4. Submit this notebook, `prompt.md`, a public GitHub repository link, a Colab link your lecturer can open, and a 4–6 sentence reflection.
5. Stop the Gradio app when finished. Anyone with the temporary share link may use the chatbot and its configured OpenAI key.